# K-Means Clustering: Palmer Penguins

In this exercise, we will use **k-means** to group Palmer penguins based on physical measurements. The dataset contains three known species, so we will set `k = 3` and compare the resulting clusters with the species labels.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import adjusted_rand_score, silhouette_score

%matplotlib inline

In [ ]:
penguins = pd.read_csv("data/penguins.csv")

print("Dataset shape:", penguins.shape)
print("Species:")
print(penguins["species"].value_counts())
penguins.head()

## Prepare the measurements

We use the four numeric body measurements as features. The species column is retained only for evaluating the clusters; it is not used to train k-means.

The features have different scales, so we standardize them before clustering.

In [ ]:
features = [
    "bill_length_mm",
    "bill_depth_mm",
    "flipper_length_mm",
    "body_mass_g",
]

model_data = penguins.dropna(subset=features + ["species"]).copy()
X = model_data[features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Rows used:", len(model_data))
print("Rows removed by missing measurement values:", len(penguins) - len(model_data))

## Fit a 3-cluster model

In [ ]:
k = 3
model = KMeans(n_clusters=k, random_state=42, n_init=10)
model_data["cluster"] = model.fit_predict(X_scaled)

print("Inertia:", round(model.inertia_, 2))
print("Silhouette score:", round(silhouette_score(X_scaled, model_data["cluster"]), 3))

## Compare clusters with species

The cluster numbers are arbitrary. A **cross-tabulation** shows which real species is most common in each cluster.

In [ ]:
cluster_species = pd.crosstab(
    model_data["cluster"],
    model_data["species"],
    margins=True,
)
cluster_species

In [ ]:
counts = pd.crosstab(model_data["cluster"], model_data["species"])
cluster_to_species = counts.idxmax(axis=1)
model_data["predicted_species"] = model_data["cluster"].map(cluster_to_species)

print("Cluster to majority-species mapping:")
print(cluster_to_species)

accuracy = (model_data["predicted_species"] == model_data["species"]).mean()
print("Majority-match accuracy:", round(accuracy * 100, 2), "%")
print("Adjusted Rand score:", round(adjusted_rand_score(model_data["species"], model_data["cluster"]), 3))

## Visualize the match

The left plot colors penguins by the actual species. The right plot colors the same observations by k-means cluster.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for species, group in model_data.groupby("species"):
    axes[0].scatter(
        group["flipper_length_mm"],
        group["body_mass_g"],
        label=species,
        alpha=0.75,
        s=45,
    )

for cluster, group in model_data.groupby("cluster"):
    axes[1].scatter(
        group["flipper_length_mm"],
        group["body_mass_g"],
        label=f"Cluster {cluster}",
        alpha=0.75,
        s=45,
    )

axes[0].set(title="Actual species", xlabel="Flipper length (mm)", ylabel="Body mass (g)")
axes[1].set(title="K-Means clusters (k=3)", xlabel="Flipper length (mm)", ylabel="Body mass (g)")
axes[0].legend()
axes[1].legend()
plt.tight_layout()
plt.show()

## Interpretation

- The crosstab shows how closely each cluster corresponds to a species.
- An adjusted Rand score of 1.0 would mean perfect agreement with the species labels, while 0.0 would be close to random agreement.
- K-means does not use the species column. The species comparison is an evaluation only.